# 06 — DQ Monitoring Logs: `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

Este notebook calcula e grava, na tabela Delta **compartilhada** `squad1/dq_monitoring_logs`, o resultado da execução das regras de qualidade de dados (DQ) sobre as duas tabelas Silver desta dupla. A tabela é compartilhada: **todas as duplas do squad1 gravam (append) nela**, cada uma reportando as regras das suas próprias tabelas.

## Regras/KPIs gerados por este notebook

**`ecommerce_itens_pedido`** (10 regras, severidade entre parênteses):
1. `tecnica_1_pk_valida` (Crítica) — chave primária do item não é nula/duplicada
2. `tecnica_2_fk_pedido_valido` (Crítica) — `id_pedido` existe na tabela de pedidos
3. `tecnica_3_fk_sku_valido` (Crítica) — SKU do produto existe na tabela de produtos
4. `tecnica_4_quantidade_valida` (Crítica) — `quantidade` > 0
5. `tecnica_5_preco_valido` (Crítica) — `preco_unitario` > 0
6. `negocio_6_desconto_nao_maior_preco` (Aviso) — `desconto_aplicado` não é maior que `preco_unitario`
7. `negocio_7_soma_itens_igual_valor_total` (Aviso, regra agregada) — soma dos itens de um pedido bate com o `valor_total` do pedido
8. `negocio_8_desconto_nao_negativo` (Aviso) — `desconto_aplicado` não é negativo
9. `negocio_9_desconto_dentro_limite` (Aviso) — `desconto_aplicado` não passa de 50% do `preco_unitario`
10. `negocio_10_pedido_tem_item` (Aviso, regra agregada) — todo pedido tem pelo menos 1 item

**`ecommerce_rastreamento_entregas`** (10 regras, severidade entre parênteses):
1. `tecnica_1_pk_valida` (Crítica) — chave primária do evento de rastreio não é nula/duplicada
2. `tecnica_2_status_valido` (Crítica) — `status_entrega` está dentro do domínio esperado
3. `tecnica_3_fk_pedido_valido` (Crítica) — `id_pedido_ecommerce` existe na tabela de pedidos
4. `tecnica_4_data_valida` (Crítica) — `dt_evento` é uma data válida
5. `tecnica_5_codigo_rastreio_valido` (Crítica) — código de rastreio não é nulo/vazio
6. `negocio_6_ordem_cronologica` (Aviso) — eventos do mesmo pedido seguem ordem cronológica coerente
7. `negocio_7_entregue_tem_evento` (Aviso, regra agregada) — pedido marcado "Entregue" tem um evento de rastreio "entregue" correspondente
8. `negocio_8_prazo_entrega_30_dias` (Aviso, regra agregada) — prazo entre "coletado" e "entregue" não passa de 30 dias
9. `negocio_9_cancelado_sem_evento` (Aviso, regra agregada) — pedido cancelado não tem evento de rastreio
10. `negocio_10_transportadora_unica` (Aviso, regra agregada) — pedido não cancelado é rastreado por uma única transportadora

**Critério de severidade:** regra Técnica = Crítica, regra de Negócio = Aviso — esse mapeamento é uma escolha desta dupla, não um padrão oficial do squad (outras duplas definem severidade por regra, caso a caso, sem fórmula fixa).

**Nota sobre os achados reais desta execução:** `tecnica_4_quantidade_valida`, `tecnica_5_preco_valido`, `negocio_6_desconto_nao_maior_preco` e `negocio_9_desconto_dentro_limite` encontraram falhas reais em `itens_pedido`. `negocio_10_pedido_tem_item` também falhou bastante (2195/2222), mas isso é um artefato de atraso de ingestão (a Bronze de `itens_pedido` só tem 129 linhas processadas até agora, bem atrás da Silver de `pedidos`, com 2222), não um defeito de dado. Em `rastreamento_entregas`, só `negocio_7_entregue_tem_evento` falha (61/61) — achado já conhecido de pedidos "Entregue" sem evento de rastreio batendo. Nenhuma linha é limpa/apagada por este notebook: tudo é apenas sinalizado (`FAIL`/`PASS`) e gravado no log.

In [0]:
# ============================================================
# CÉLULA 1 — SETUP
# DQ MONITORING LOGS - ITENS_PEDIDO e RASTREAMENTO_ENTREGAS
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net"

caminhos = {
    "silver_itens_pedido": f"{base_squad1}/dupla1/silver/ecommerce_itens_pedido",
    "silver_rastreamento": f"{base_squad1}/dupla1/silver/ecommerce_rastreamento_entregas",
    "silver_pedidos": f"{base_squad1}/silver/ecommerce_pedidos",
    "dq_logs": f"{base_squad1}/dq_monitoring_logs",
}

print("Credenciais OAuth preparadas.")
print("DQ Logs (compartilhado):", caminhos["dq_logs"])

## Etapa 1 — Setup

Credenciais OAuth para o ADLS (padrão `dbutils.secrets.get`) e caminhos usados pelo notebook: as duas tabelas Silver desta dupla, a tabela de `pedidos` (usada pelas regras agregadas) e o caminho da tabela compartilhada `dq_monitoring_logs`, na raiz do container `squad1` (sem prefixo `dupla1/`, porque é compartilhada entre todas as duplas).

In [0]:
# ============================================================
# CÉLULA 2 — REGRAS DE QUALIDADE POR LINHA (com severidade)
# ============================================================

# itens_pedido: colunas booleanas já gravadas na Silver
REGRAS_ITENS_PEDIDO = {
    "tecnica_1_pk_valida":              "Crítica",
    "tecnica_2_fk_pedido_valido":       "Crítica",
    "tecnica_3_fk_sku_valido":          "Crítica",
    "tecnica_4_quantidade_valida":      "Crítica",
    "tecnica_5_preco_valido":           "Crítica",
    "negocio_6_desconto_nao_maior_preco": "Aviso",
    "negocio_8_desconto_nao_negativo":    "Aviso",
    "negocio_9_desconto_dentro_limite":   "Aviso",
}

# rastreamento_entregas: colunas booleanas já gravadas na Silver
REGRAS_RASTREAMENTO = {
    "tecnica_1_pk_valida":              "Crítica",
    "tecnica_2_status_valido":          "Crítica",
    "tecnica_3_fk_pedido_valido":       "Crítica",
    "tecnica_4_data_valida":            "Crítica",
    "tecnica_5_codigo_rastreio_valido": "Crítica",
    "negocio_6_ordem_cronologica":      "Aviso",
}

print("Regras itens_pedido:", list(REGRAS_ITENS_PEDIDO.keys()))
print("Regras rastreamento:", list(REGRAS_RASTREAMENTO.keys()))

## Etapa 2 — Regras de qualidade por linha

Dicionários `REGRAS_ITENS_PEDIDO` e `REGRAS_RASTREAMENTO` mapeando cada regra booleana já calculada na Silver (colunas `tecnica_*`/`negocio_*`) para sua severidade (`Crítica` ou `Aviso`). Só entram aqui as regras **por linha** — as regras agregadas (por pedido) são tratadas separadamente na Etapa 4, porque não existem como coluna booleana na Silver.

In [0]:
# ============================================================
# CÉLULA 3 — CALCULAR FALHAS POR REGRA (linhas)
# ============================================================

run_id = str(uuid.uuid4())
agora = datetime.now(timezone.utc)

def calcular_falhas_por_regra(df, nome_tabela, regras_dict):
    total = df.count()
    linhas = []
    for regra, severidade in regras_dict.items():
        qtd_falhas = df.filter(F.col(regra) == False).count()
        status = "FAIL" if qtd_falhas > 0 else "PASS"
        linhas.append(
            (run_id, nome_tabela, regra, status, severidade, qtd_falhas, total, agora)
        )
        print(f"[{nome_tabela}] {regra} -> {status} | falhas: {qtd_falhas}/{total}")
    return linhas

df_silver_itens = spark.read.format("delta").options(**adls_options).load(caminhos["silver_itens_pedido"])
df_silver_rastreamento = spark.read.format("delta").options(**adls_options).load(caminhos["silver_rastreamento"])

linhas_logs = []
linhas_logs += calcular_falhas_por_regra(df_silver_itens, "ecommerce_itens_pedido", REGRAS_ITENS_PEDIDO)
linhas_logs += calcular_falhas_por_regra(df_silver_rastreamento, "ecommerce_rastreamento_entregas", REGRAS_RASTREAMENTO)

## Etapa 3 — Cálculo de falhas por regra (linhas)

`calcular_falhas_por_regra` lê cada tabela Silver, conta o total de linhas e, para cada regra do dicionário, conta quantas linhas têm aquela coluna booleana `False`. Cada regra gera 1 linha de log (`run_id`, tabela, regra, status PASS/FAIL, severidade, qtd. de falhas, total de linhas, timestamp). `run_id` é um UUID único para esta execução — é o que permite, depois, isolar e conferir só os logs gravados por este run dentro da tabela compartilhada.

In [0]:
# ============================================================
# CÉLULA 4 — REGRAS AGREGADAS (por pedido)
# ============================================================

df_silver_pedidos = spark.read.format("delta").options(**adls_options).load(caminhos["silver_pedidos"])

# --- ITENS_PEDIDO: Regra 7 (soma dos itens == valor_total do pedido) ---
soma_por_pedido = (df_silver_itens
    .withColumn("valor_item", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"))
    .groupBy("id_pedido").agg(F.sum("valor_item").alias("soma_calculada"))
    .join(df_silver_pedidos.select("id_pedido", "valor_total"), on="id_pedido", how="left"))

total_pedidos_itens = soma_por_pedido.count()
divergentes = soma_por_pedido.filter(
    F.col("valor_total").isNull() | (F.abs(F.col("soma_calculada") - F.col("valor_total")) > 0.01)
).count()
status = "FAIL" if divergentes > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_itens_pedido", "negocio_7_soma_itens_igual_valor_total", status, "Aviso", divergentes, total_pedidos_itens, agora))
print(f"[itens_pedido] negocio_7_soma_itens_igual_valor_total -> {status} | falhas: {divergentes}/{total_pedidos_itens}")

# --- ITENS_PEDIDO: Regra 10 (todo pedido tem >=1 item) ---
total_pedidos = df_silver_pedidos.count()
pedidos_com_item = df_silver_itens.select("id_pedido").distinct()
pedidos_sem_item = df_silver_pedidos.select("id_pedido").distinct().subtract(pedidos_com_item).count()
status = "FAIL" if pedidos_sem_item > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_itens_pedido", "negocio_10_pedido_tem_item", status, "Aviso", pedidos_sem_item, total_pedidos, agora))
print(f"[itens_pedido] negocio_10_pedido_tem_item -> {status} | falhas: {pedidos_sem_item}/{total_pedidos}")

# --- RASTREAMENTO: Regra 7 (pedido Entregue tem evento 'entregue') ---
ids_com_evento_entregue = set(r["id_pedido_ecommerce"] for r in
    df_silver_rastreamento.filter(F.col("status_entrega") == "entregue").select("id_pedido_ecommerce").distinct().collect())
pedidos_entregues = df_silver_pedidos.filter(F.col("status_pedido") == "Entregue").select("id_pedido").distinct()
total_entregues = pedidos_entregues.count()
sem_evento_entregue = pedidos_entregues.filter(~F.col("id_pedido").isin(ids_com_evento_entregue)).count()
status = "FAIL" if sem_evento_entregue > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_7_entregue_tem_evento", status, "Aviso", sem_evento_entregue, total_entregues, agora))
print(f"[rastreamento] negocio_7_entregue_tem_evento -> {status} | falhas: {sem_evento_entregue}/{total_entregues}")

# --- RASTREAMENTO: Regra 8 (<=30 dias entre coletado e entregue) ---
eventos_coletado = df_silver_rastreamento.filter(F.col("status_entrega") == "coletado").groupBy("id_pedido_ecommerce").agg(F.min("dt_evento").alias("dt_coletado"))
eventos_entregue = df_silver_rastreamento.filter(F.col("status_entrega") == "entregue").select("id_pedido_ecommerce", F.col("dt_evento").alias("dt_entregue"))
prazos = eventos_coletado.join(eventos_entregue, on="id_pedido_ecommerce", how="inner").withColumn("dias_entrega", F.datediff("dt_entregue", "dt_coletado"))
total_prazos = prazos.count()
fora_do_prazo = prazos.filter(F.col("dias_entrega") > 30).count()
status = "FAIL" if fora_do_prazo > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_8_prazo_entrega_30_dias", status, "Aviso", fora_do_prazo, total_prazos, agora))
print(f"[rastreamento] negocio_8_prazo_entrega_30_dias -> {status} | falhas: {fora_do_prazo}/{total_prazos}")

# --- RASTREAMENTO: Regra 9 (pedido cancelado não pode ter evento) ---
pedidos_cancelados = df_silver_pedidos.filter(F.col("status_pedido") == "Cancelado").select("id_pedido").distinct()
total_cancelados = pedidos_cancelados.count()
ids_cancelados_com_evento = (df_silver_rastreamento.select("id_pedido_ecommerce").distinct()
    .join(pedidos_cancelados.withColumnRenamed("id_pedido", "id_pedido_ecommerce"), on="id_pedido_ecommerce", how="inner").count())
status = "FAIL" if ids_cancelados_com_evento > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_9_cancelado_sem_evento", status, "Aviso", ids_cancelados_com_evento, total_cancelados, agora))
print(f"[rastreamento] negocio_9_cancelado_sem_evento -> {status} | falhas: {ids_cancelados_com_evento}/{total_cancelados}")

# --- RASTREAMENTO: Regra 10 (pedido não cancelado tem só 1 transportadora) ---
ids_cancelados_geral = set(r["id_pedido"] for r in pedidos_cancelados.collect())
transportadoras_por_pedido = (df_silver_rastreamento
    .filter(~F.col("id_pedido_ecommerce").isin(ids_cancelados_geral))
    .groupBy("id_pedido_ecommerce").agg(F.countDistinct("id_transportadora").alias("qtd_transportadoras")))
total_nao_cancelados = transportadoras_por_pedido.count()
inconsistentes = transportadoras_por_pedido.filter(F.col("qtd_transportadoras") > 1).count()
status = "FAIL" if inconsistentes > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_10_transportadora_unica", status, "Aviso", inconsistentes, total_nao_cancelados, agora))
print(f"[rastreamento] negocio_10_transportadora_unica -> {status} | falhas: {inconsistentes}/{total_nao_cancelados}")

print(f"\nTotal de regras no log: {len(linhas_logs)}")

## Etapa 4 — Regras agregadas (por pedido)

Regras que não podem ser calculadas linha a linha, porque dependem de agregações por `id_pedido` (soma dos itens vs. valor total do pedido, pedido sem item, pedido entregue sem evento de rastreio, prazo de entrega, pedido cancelado com evento, pedido com mais de uma transportadora). Cada uma é calculada manualmente e adicionada à mesma lista `linhas_logs` da Etapa 3, para ir junto no mesmo lote de gravação.

In [0]:
# ============================================================
# CÉLULA 5 — MONTAR SCHEMA E DATAFRAME DE LOGS
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType
)

schema_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", LongType(), False),
    StructField("qtd_registros_total", LongType(), False),
    StructField("timestamp_execucao", TimestampType(), False),
])

df_novo = spark.createDataFrame(linhas_logs, schema=schema_logs)

print("===== LOGS QUE SERÃO GRAVADOS =====")
display(df_novo)

## Etapa 5 — Schema e DataFrame de logs

Schema explícito (`schema_logs`) com os 8 campos do log: `run_id`, `tabela`, `regra`, `status`, `severidade`, `qtd_registros_falhos`, `qtd_registros_total`, `timestamp_execucao`. O campo `arquivo_origem`, que aparece em registros de outras duplas, é omitido aqui de propósito (segue o schema mais simples usado como referência) — ele aparece como `null` nas nossas linhas automaticamente, graças ao `mergeSchema` usado na gravação.

In [0]:
# ============================================================
# CÉLULA 6 — GRAVAR NO DQ_MONITORING_LOGS (COMPARTILHADO)
# ============================================================

(df_novo.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .save(caminhos["dq_logs"]))

print(f"[OK] {len(linhas_logs)} logs gravados em dq_monitoring_logs (run_id={run_id}).")

## Etapa 6 — Gravação no `dq_monitoring_logs` (compartilhado)

Escrita em modo `append` com `mergeSchema=true`, na tabela Delta compartilhada por todo o squad1. **Nunca** é usado `overwrite` aqui — cada execução soma suas linhas ao histórico de todas as duplas; é por isso que o notebook Gold (07) precisa deduplicar por execução mais recente antes de agregar.

In [0]:
# ============================================================
# CÉLULA 7 — VALIDAÇÃO FINAL
# ============================================================

df_logs_final = spark.read.format("delta").options(**adls_options).load(caminhos["dq_logs"])

df_nossos_logs = (df_logs_final
    .filter(F.col("run_id") == run_id)
    .orderBy("tabela", "regra"))

print("===== NOSSOS LOGS NESTA EXECUÇÃO =====")
display(df_nossos_logs)

print(f"\nTotal de linhas na tabela compartilhada dq_monitoring_logs: {df_logs_final.count()}")

## Etapa 7 — Validação final

Relê a tabela compartilhada, filtra apenas as linhas do `run_id` desta execução para confirmar que a gravação funcionou como esperado, e imprime o total de linhas na tabela inteira (soma de todas as duplas) como referência de contexto.